# 高级索引与重复位置更新

学习目标：根据逐对坐标或矩形区域选择索引写法，解释混合索引的结果形状，并正确累加重复位置。

前置知识：数组形状、基本切片、整数数组索引、广播、视图与副本、ufunc。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例均使用本章的小数组，后续单元沿用首次导入的 np。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 按坐标提取读数

二维表的行表示三次观测，列表示四个传感器。现在分别取第一次观测的第二个传感器、第三次观测的第四个传感器。

给两个轴各提供一个同形整数数组时，索引按相同位置配对。rows 和 columns 中第一个数形成一个坐标，第二个数形成另一个坐标；不是取这两行与两列组成的全部区域。

In [1]:
import numpy as np

readings = np.array([[11, 12, 13, 14], [21, 22, 23, 24], [31, 32, 33, 34]], dtype=np.int16)
rows = np.array([0, 2])
columns = np.array([1, 3])
selected = readings[rows, columns]

print(selected)  # [12 34]，分别来自坐标 (0, 1) 和 (2, 3)。
print(selected.shape, selected.dtype)  # (2,) int16，每对坐标得到一个值。

[12 34]
(2,) int16


## 2 选择矩形区域

### 2.1 索引数组的广播

多个整数索引数组会一起广播。二维数组的两个轴都由这些索引指定时，输出形状就是索引广播后的形状。

要取得两行与三列的所有组合，可以把行索引变成 (2, 1)，让它与形状 (3,) 的列索引广播成 (2, 3)。下面重新给出输入，行和列的选择互相独立。

In [2]:
readings = np.array([[11, 12, 13, 14], [21, 22, 23, 24], [31, 32, 33, 34]], dtype=np.int16)
rows = np.array([0, 2])
columns = np.array([0, 1, 3])
rectangle = readings[rows[:, np.newaxis], columns]

print(rows[:, np.newaxis].shape, columns.shape)  # (2, 1) (3,)。
print(rectangle)  # 两行分别为 [11 12 14] 和 [31 32 34]。
print(rectangle.shape, rectangle.dtype)  # (2, 3) int16。
print(np.shares_memory(readings, rectangle))  # False，高级索引读取得到副本。

(2, 1) (3,)
[[11 12 14]
 [31 32 34]]
(2, 3) int16
False


### 2.2 ix_ 表达各轴的所有组合

np.ix_() 接收各轴的一维整数序列，为它们生成适合广播的索引。需要“选中这些行、这些列”的矩形结果时，这种写法比手动增加轴更直接。

以下沿用上一单元的 readings、rows 和 columns，比较两种写法。

下图另用行 [0, 2]、列 [1, 3] 对照两种选择：逐对取两个位置，矩形选取四个位置。

![下图另用行 [0, 2]、列 [1, 3] 对照两种选择：逐对取两个位置，矩形选取四个位置。](image/09-paired-grid.png)

In [3]:
row_grid, column_grid = np.ix_(rows, columns)
rectangle = readings[row_grid, column_grid]

print(row_grid.shape, column_grid.shape)  # (2, 1) (1, 3)。
print(rectangle)  # 两行分别为 [11 12 14] 和 [31 32 34]。
print(np.array_equal(rectangle, readings[rows[:, None], columns]))  # True。

(2, 1) (1, 3)
[[11 12 14]
 [31 32 34]]
True


### 2.3 无法广播的索引

两个一维索引长度分别为 2 和 3 时，不能直接逐对匹配。此时会触发 IndexError，而不会自动变成矩形选择。

下面保留这个预期失败，用来区分“坐标配对”和“所有组合”。

In [4]:
readings = np.array([[11, 12, 13, 14], [21, 22, 23, 24], [31, 32, 33, 34]])

# 预期 IndexError：两组整数索引的形状 (2,) 与 (3,) 无法广播成共同形状。
readings[[0, 2], [0, 1, 3]]

IndexError: shape mismatch: indexing arrays could not be broadcast together with shapes (2,) (3,) 

## 3 读取副本与直接赋值

高级索引读取返回副本，修改读取结果不会改变原数组。把索引直接放在赋值左侧，则会修改原数组选中的位置；赋值内容要能广播到选中结果的形状。

下面先修改保存下来的读取结果，再直接修改原数组，观察二者区别。

In [5]:
readings = np.array([[11, 12, 13], [21, 22, 23]], dtype=np.int16)
selected = readings[[0, 1], [1, 2]]
selected[:] = 99
print(readings)  # 原数组仍为 [11 12 13]、[21 22 23]。
print(selected, selected.shape, selected.dtype)  # [99 99] (2,) int16。
print(np.shares_memory(readings, selected))  # False。

readings[[0, 1], [1, 2]] = [50, 60]
print(readings)  # 两行变为 [11 50 13]、[21 22 60]。
print(selected)  # [99 99]，之前的副本不会随原数组变化。

[[11 12 13]
 [21 22 23]]
[99 99] (2,) int16
False
[[11 50 13]
 [21 22 60]]
[99 99]


## 4 混合索引的结果轴

### 4.1 相邻的高级索引

切片与多个高级索引混合时，既要看索引广播后的形状，也要看保留的基本切片轴。

高级索引相邻时，它们广播得到的轴放在原来那组轴所在的位置。下面 cube 的轴依次表示 2 个批次、3 次观测、4 个传感器；两个相邻索引逐对选取三个“观测、传感器”坐标，批次轴保留在前面。

In [6]:
cube = np.arange(24, dtype=np.int16).reshape(2, 3, 4)
selected = cube[:, [0, 1, 2], [1, 2, 3]]

print(selected)  # 两行分别为 [1 6 11] 和 [13 18 23]。
print(selected.shape, selected.dtype)  # (2, 3) int16：批次、坐标对。
print(np.shares_memory(cube, selected))  # False。

[[ 1  6 11]
 [13 18 23]]
(2, 3) int16
False


### 4.2 被切片隔开的高级索引

高级索引被切片、Ellipsis 或 newaxis 隔开时，广播得到的高级索引轴放到结果最前面，基本切片保留的轴随后排列。

下面用两对“批次、传感器”坐标取值，中间保留全部观测。每一行对应一对坐标，每一列对应一次观测。结果虽然也为 (2, 3)，两个轴的含义与上一例不同，需核对元素来源。

下图对应两段 cube 示例；即使 shape 相同，也要辨认每条轴代表什么。

![下图对应两段 cube 示例；即使 shape 相同，也要辨认每条轴代表什么。](image/09-advanced-axes.png)

In [7]:
cube = np.arange(24, dtype=np.int16).reshape(2, 3, 4)
selected = cube[[0, 1], :, [1, 3]]

print(selected)  # 两行分别为 [1 5 9] 和 [15 19 23]。
print(selected.shape, selected.dtype)  # (2, 3) int16：坐标对、观测。
print(np.array_equal(selected[0], cube[0, :, 1]))  # True，第一对是批次 0、传感器 1。
print(np.array_equal(selected[1], cube[1, :, 3]))  # True，第二对是批次 1、传感器 3。

[[ 1  5  9]
 [15 19 23]]
(2, 3) int16
True
True


## 5 重复位置的更新

### 5.1 += 不等于逐次累加

用整数索引数组执行 +=，会先读取选中值、计算，再写回；重复出现的位置不会因此按出现次数累加。

下面位置 1 出现两次，但两次读取的初值都是 0，增加 1 后写回，结果仍为 1。这个行为不适合统计每个位置收到多少次事件。

In [8]:
positions = np.array([1, 1, 3])
counts = np.zeros(4, dtype=np.int64)
counts[positions] += 1

print(counts)  # [0 1 0 1]，位置 1 没有得到两次累加。
print(counts.shape, counts.dtype)  # (4,) int64。

[0 1 0 1]
(4,) int64


### 5.2 add.at 逐次更新原数组

np.add.at() 对指定位置执行无缓冲的原地加法，同一位置出现多次时，每一次贡献都会参与累计。第三个参数可以是一个值，也可以是能广播到所选位置形状的数组。

普通高级索引赋值在同一元素被反复赋不同值时，不应依赖写入次序。需要累计时，直接表达累计规则，不把重复赋值当作求和。

In [9]:
positions = np.array([1, 1, 3])
counts = np.zeros(4, dtype=np.int64)
np.add.at(counts, positions, 1)
print(counts)  # [0 2 0 1]，位置 1 正确累计两次。

weights = np.array([2, 3, 4], dtype=np.int64)
totals = np.zeros(4, dtype=np.int64)
np.add.at(totals, positions, weights)
print(totals)  # [0 5 0 4]，位置 1 收到 2 + 3。
print(totals.shape, totals.dtype)  # (4,) int64。

[0 2 0 1]
[0 5 0 4]
(4,) int64


## 6 综合应用：汇总通道事件

五个事件分别给三个通道贡献不同的计数。通道编号从 0 开始，输入位置均合法；一个通道可能出现多次。

先按编号累计，再按指定的展示顺序读取结果。累计使用 add.at，展示使用普通整数索引读取，二者用途不同。

In [10]:
channels = np.array([0, 2, 0, 1, 2])
increments = np.array([2, 5, 3, 4, 1], dtype=np.int64)
totals = np.zeros(3, dtype=np.int64)
np.add.at(totals, channels, increments)
displayed = totals[[2, 0, 1]]

print(totals)  # [5 4 6]，手算分别为 2 + 3、4、5 + 1。
print(displayed, displayed.shape, displayed.dtype)  # [6 5 4] (3,) int64。
print(np.array_equal(totals, np.array([5, 4, 6])))  # True，与手算累计一致。
print(np.shares_memory(totals, displayed))  # False，展示结果是副本。

[5 4 6]
[6 5 4] (3,) int64
True
False


## 7 选学：按各行的不同位置取值

np.take_along_axis() 可以沿一个轴，根据每个切片各自的索引取值。下面每一行有两个指定列位置，axis=1 表示沿列轴选择。

indices 与输入的维数应相同；选取轴以外的对应维度需要能广播。与此相比，np.take() 对各个切片使用同一组索引。这里显式指定 axis，避免依赖默认值。

In [11]:
table = np.array([[10, 20, 30], [40, 50, 60]], dtype=np.int16)
indices = np.array([[2, 0], [0, 1]])
selected = np.take_along_axis(table, indices, axis=1)

print(selected)  # 两行分别为 [30 10]、[40 50]，各行使用自己的列索引。
print(selected.shape, selected.dtype)  # (2, 2) int16。
print(np.take(table, [2, 0], axis=1))  # 两行分别为 [30 10]、[60 40]，使用相同列索引。

[[30 10]
 [40 50]]
(2, 2) int16
[[30 10]
 [60 40]]


## 本章小结

（1）多个整数索引先广播、再逐对取值；矩形区域可用 ix_ 表达各轴的所有组合。

（2）高级索引读取是副本，直接索引赋值仍会修改原数组；混合索引需同时检查数值与轴含义。

（3）重复位置的 += 不代表逐次累计；需要每次贡献都写入时，可以使用 add.at。

（4）take_along_axis 适合各行或各切片使用不同位置的取值任务。

## 练习

（1）先预测两种索引结果的值、形状与共享关系，再运行核对。用“坐标对”和“所有组合”解释差别。

In [12]:
table = np.array([[10, 11, 12], [20, 21, 22], [30, 31, 32]])
rows = np.array([0, 2])
columns = np.array([0, 2])

# 先写下预测，再运行；不要只比较元素数量。
print(table[rows, columns])
print(table[np.ix_(rows, columns)])
# 在此补充 shape 与 shares_memory 检查。

[10 32]
[[10 12]
 [30 32]]


（2）任务甲：选择坐标 (0, 1)、(2, 3) 的两个值。任务乙：保留第 0、2 行与第 1、3 列的全部交叉值，并保持二维。分别选择索引方法并解释理由。

In [13]:
table = np.array([[11, 12, 13, 14], [21, 22, 23, 24], [31, 32, 33, 34]])

# 在此实现两个任务，并说明为什么不能用同一种逐对索引满足两者。
# 检查：甲为 [12 34]、shape (2,)；乙为 [[12, 14], [32, 34]]、shape (2, 2)。

（3）把四个事件的贡献累计到三个位置。要求同一位置收到的每一次贡献都保留，并解释普通 += 为什么不满足这个条件。

In [14]:
positions = np.array([0, 2, 0, 2])
increments = np.array([1, 3, 5, 7], dtype=np.int64)
totals = np.zeros(3, dtype=np.int64)

# 在此选择更新方法；检查：结果为 [6 0 10]、shape (3,)、dtype int64。
# 在注释中说明索引中的重复位置及需要的累计规则。

（4）下面两个混合索引分别保留哪个原始轴？先预测每个输出轴的含义，再运行。选择一个输出元素，写出它在 cube 中的完整坐标核对。

In [15]:
cube = np.arange(24).reshape(2, 3, 4)
adjacent = cube[:, [0, 2], [1, 3]]
separated = cube[[0, 1], :, [1, 3]]

# 在此记录预测，并写出一个输出元素与原数组的对应坐标。
print(adjacent, adjacent.shape)
print(separated, separated.shape)

[[ 1 11]
 [13 23]] (2, 2)
[[ 1  5  9]
 [15 19 23]] (2, 3)


## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | [Indexing on ndarrays](https://numpy.org/doc/2.5/user/basics.indexing.html) 的 Integer array indexing（广播与坐标配对）、Combining advanced and basic indexing（轴位置）、Assigning values to indexed arrays（赋值、缓冲与重复位置次序）；[ix_](https://numpy.org/doc/2.5/reference/generated/numpy.ix_.html) 的参数与交叉选择；[ufunc.at](https://numpy.org/doc/2.5/reference/generated/numpy.ufunc.at.html) 的无缓冲更新与重复索引；[take_along_axis](https://numpy.org/doc/2.5/reference/generated/numpy.take_along_axis.html) 的维数、广播与 axis；[take](https://numpy.org/doc/2.5/reference/generated/numpy.take.html) 的按轴取值；[shares_memory](https://numpy.org/doc/2.5/reference/generated/numpy.shares_memory.html) 的重叠判断；[array_equal](https://numpy.org/doc/2.5/reference/generated/numpy.array_equal.html) 的形状与元素相等判断。 |